# Comparing the models in t-SNE space

A visual postscript to the defence sections: the trained models — DNN, AT-rFGSM, AT-rFGSM with weighted loss, AT-imax, and the RBF model — are compared by extracting their internal features and projecting them with **t-SNE**, to see how differently each model organises the same data.

**Where this fits:** the models compared here are built in `02-detectors/` and defended/attacked in `03-attacks/` and this section; the notebook originally sat with the mal_adv5 contrastive work but belongs to the model family above.

*Kept as its own notebook from `analyzing_models_tsne.ipynb` (originally in `mal_adv5`), lightly structured. Outputs cleared.*

## Step 1 — Setup and clean evaluation of every model

# Loading data

In [ ]:
download_links = [
                  'https://drive.google.com/uc?id=1-8lJXLdAl_4NdDwzw9kFML0aiOCTrI9f',
                  'https://drive.google.com/uc?id=1-OHACrNCt0yKBbdqQPVfNZcjKt5_jxKD',
                  'https://drive.google.com/uc?id=1-KeXJXtU1_6m9JOhormeVwigy0myX3HL',
                  'https://drive.google.com/uc?id=1-13RDdZqnrNkdHg3D8PC5KI0CZREwlsz',
                  'https://drive.google.com/uc?id=1-8LjsCdzKh6asxCFsYLiQZbSEXXKSQBP',

]

import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/combined_model_adam.pth'

]

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time
from sklearn.manifold import TSNE
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    #os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()
print('the proportion of malwares : ', n_mal/(n_mal+n_ben))

# Combine features and labels into datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
del train_dataset, val_dataset, test_dataset, y_train, y_val, y_test

In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
model_evaluation(model_DNN, test_loader, device)

In [ ]:
model_evaluation(model_AT_rFGSM,test_loader,device)

In [ ]:
model_evaluation(model_AT_rFGSM_weightedLoss,test_loader,device)

In [ ]:
model_evaluation(model_AT_imax,test_loader,device)

In [ ]:
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model_gaussian_1000 = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000 = model_gaussian_1000.to(device)

# Load the model state dictionary
model_gaussian_1000.load_state_dict(torch.load('/content/best_model_gaussian_1000.pth', map_location=torch.device(device)))

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 1144))
model_gaussian_1000_nonremoval = RBFModel(1000, 2, all_centers, [sigma], kernel)
model_gaussian_1000_nonremoval = model_gaussian_1000_nonremoval.to(device)

# Load the model state dictionary
model_gaussian_1000_nonremoval.load_state_dict(torch.load('/content/best_model_gaussian_1000_nonremoval.pth', map_location=torch.device(device)))

## Step 2 — Extracting features and running t-SNE

# compare models

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def extract_features(model, dataloader):
    model.eval()  # Set the model to evaluation mode
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            logits, last_features, second_last_features = model(inputs.to(torch.float32))
            features.append(second_last_features.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels

def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels, model_name):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title(f't-SNE of Last Layer Features - {model_name}')

    # Density plot
    #sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, ax=ax2)
    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    #sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax2, linewidths=1)
    ax2.set_title(f'Density Plot - {model_name}')

    plt.show()

In [ ]:
class MalwareDetectionModel2(nn.Module):
    def __init__(self, input_size=10000, hidden_1_size=200, hidden_2_size=200, num_labels=2, dropout_prob=0.6):
        super(MalwareDetectionModel2, self).__init__()

        self.input_size = input_size
        self.hidden_1_size = hidden_1_size
        self.hidden_2_size = hidden_2_size
        self.num_labels = num_labels
        self.dropout_prob = dropout_prob

        self.fc1 = nn.Linear(input_size, hidden_1_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_1_size, hidden_2_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_prob)
        self.fc3 = nn.Linear(hidden_2_size, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        second_last_features = self.fc2(x)
        x = self.relu2(second_last_features)
        last_features = self.dropout2(x)
        logits = self.fc3(last_features)
        return logits, last_features, second_last_features

In [ ]:
# For multiple models
model_paths = ['model_DNN_drebin_best.pth', 'model_AT_rFGSM.pth', 'model_AT_rFGSM_weightedLoss.pth', 'best_model_AT_imax.pth']

model = MalwareDetectionModel2().to(device)
for model_path in model_paths:
    model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))
    model.eval()

    # Extract features from validation set
    features, labels = extract_features(model, test_loader)

    # Apply t-SNE
    tsne_results = apply_tsne(features)

    # Plot joint plot
    plot_tsne_and_density(tsne_results, labels, model_path)

In [ ]:
class RBFModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, init_centers, init_sigmas, kernel):
        super(RBFModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.centers = nn.Parameter(torch.Tensor(init_centers))
        self.sigmas = nn.Parameter(torch.Tensor(init_sigmas))
        self.kernel = kernel
        # Linear layer for output
        self.linear = nn.Linear(hidden_dim, output_dim)

    def gaussian(self, x, c, sigma):
        return torch.exp(-torch.sum((x[:, None, :] - c) ** 2, dim=-1) / (2 * sigma ** 2))

    def laplacian(self, x, c, sigma):
        return torch.exp(-torch.sum(torch.abs(x[:, None, :] - c) , dim=-1) / sigma)

    def forward(self, x):
      if self.kernel == 'gaussian':
        radial_out = self.gaussian(x, self.centers, self.sigmas)
      elif self.kernel == 'laplacian':
        radial_out = self.laplacian(x, self.centers, self.sigmas)
      else:
        raise ValueError("Invalid kernel type. Choose 'gaussian' or 'laplacian'.")

      output = self.linear(radial_out.to(torch.float32))
      return output,output,radial_out

In [ ]:
non_removal_features = False
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 10000))
model = RBFModel(1000, 2, all_centers, [sigma], kernel)
model = model.to(device)

model_path = 'best_model_gaussian_1000.pth'

model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))

model.eval()

# Extract features from validation set
features, labels = extract_features(model, test_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels, model_path)

In [ ]:
non_removal_mask.shape

In [ ]:
def extract_features3(model, data_loader):
    model.eval()
    all_last_features = []
    all_second_last_features = []
    all_labels = []

    with torch.no_grad():
        for data, labels in data_loader:
            data = data.to(torch.float32).to(device)
            labels = labels.to(device)
            logits, last_features, second_last_features = model(data[:,non_removal_mask])
            all_last_features.append(second_last_features.cpu())
            all_labels.append(labels.cpu())

    return torch.cat(all_last_features), torch.cat(all_labels)

In [ ]:
non_removal_features = True
non_removal_mask = torch.logical_not(removal_array).to('cpu')
sigma = 4.15
kernel = 'gaussian'
all_centers = torch.rand((1000, 1144))
model = RBFModel(1000, 2, all_centers, [sigma], kernel)
model = model.to(device)

model_path = '/content/best_model_gaussian_1000_nonremoval.pth'

model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))
model.eval()
features, labels = extract_features3(model, test_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels, model_path)

## Step 3 — Comparing the models' outputs

# comapring outputs of models

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            outputs = model(inputs.to(torch.float32))
            features.append(outputs.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels


def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels, model_name):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title(f't-SNE of Last Layer Features - {model_name}')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title(f'Density Plot - {model_name}')

    plt.show()

In [ ]:
class MalwareDetectionModel2(nn.Module):
    def __init__(self, input_size=10000, hidden_1_size=200, hidden_2_size=200, num_labels=2, dropout_prob=0.6):
        super(MalwareDetectionModel2, self).__init__()

        self.input_size = input_size
        self.hidden_1_size = hidden_1_size
        self.hidden_2_size = hidden_2_size
        self.num_labels = num_labels
        self.dropout_prob = dropout_prob

        self.fc1 = nn.Linear(input_size, hidden_1_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_1_size, hidden_2_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_prob)
        self.fc3 = nn.Linear(hidden_2_size, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        second_last_features = self.fc2(x)
        x = self.relu2(second_last_features)
        last_features = self.dropout2(x)
        logits = self.fc3(last_features)
        return logits

In [ ]:
combined_model = torch.load('combined_model_adam.pth', map_location=torch.device(device))
combined_model.eval()

# Extract features from validation set
features, labels = extract_features(combined_model.cpu(), test_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels,'combined_model_adam')